In [0]:
spark

In [0]:
raw_df = spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/datasets/raw/raw_volume/accepted_2007_to_2018Q4.csv.gz")

In [0]:
raw_df.count()

In [0]:
raw_df.display()

In [0]:
raw_df.createOrReplaceTempView("lending_club_data")

In [0]:
%sql
select * from lending_club_data

In [0]:
spark.sql("select * from lending_club_data")

In [0]:
raw_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
new_df = raw_df.withColumn("name_sha2", sha2(concat_ws("||", *["emp_title","emp_length","home_ownership","annual_inc","zip_code","addr_state","grade","sub_grade","verification_status"]),256))

In [0]:
new_df.display()

In [0]:
new_df.show()

In [0]:
new_df.createOrReplaceTempView("new_table")

In [0]:
spark.sql("select * from new_table")

In [0]:
spark.sql("select count(*) from new_table").show()

In [0]:
%sql
select count(*) from new_table

In [0]:
new_df.count()

In [0]:
spark.sql("select name_sha2, count(*) as total_count from new_table group by name_sha2 having total_count > 1 order by total_count desc").display()

In [0]:
spark.sql("select * from new_table where name_sha2 like 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855%'").display()

In [0]:
%fs

ls

In [0]:
new_df.printSchema()

In [0]:
spark.sql("""select name_sha2 as member_id, emp_title, emp_length, home_ownership, annual_inc, addr_state,zip_code,'USA' AS COUNTRY, grade, sub_grade, verification_status, tot_hi_cred_lim, application_type, annual_inc_joint, verification_status_joint from new_table""").repartition(1).write\
    .option("header", True)\
    .format("csv")\
    .mode("overwrite")\
    .ption("path","/Volumes/datasets/raw/raw_volumestomers_data_csv")\
.save()


In [0]:
spark.sql("""select name_sha2 as member_id, emp_title, emp_length, home_ownership, annual_inc, addr_state,zip_code,'USA' AS COUNTRY, grade, sub_grade, verification_status, tot_hi_cred_lim, application_type, annual_inc_joint, verification_status_joint from new_table""").repartition(1).write\
    .option("header", True)\
    .format("delta")\
    .mode("overwrite")\
    .option("path","/Volumes/datasets/raw/raw_volume/raw/delta/customer_delta")\
    .save()


In [0]:
%fs

ls

In [0]:
customer_df = spark.read\
    .format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load("/Volumes/datasets/raw/raw_volume/customers_data_csv/")

In [0]:
customer_df.display()

# Loans Data

In [0]:
customer_df.printSchema()

In [0]:
spark.sql(""" Select id as Loan_id, name_sha2 as member_id,loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose,title from new_table""").display()

In [0]:
spark.sql(""" Select id as Loan_id, name_sha2 as member_id,loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose,title from new_table""").repartition(1).write\
    .option("header", True)\
    .format("csv")\
    .mode("overwrite")\
    .option("path","/Volumes/datasets/raw/raw_volume/loans_data_csv").save()

In [0]:
loans_df = spark.read\
    .format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load("/Volumes/datasets/raw/raw_volume/loans_data_csv/")
    

In [0]:
loans_df.display()

# Loan Repayments

In [0]:
spark.sql("""Select id as loan_id, total_rec_prncp, total_rec_int, total_rec_late_fee,total_pymnt, last_pymnt_amnt, next_pymnt_d, last_credit_pull_d, next_pymnt_d from new_table""").display()


In [0]:
spark.sql("""Select id as loan_id, total_rec_prncp, total_rec_int, total_rec_late_fee,total_pymnt, last_pymnt_amnt, next_pymnt_d, last_credit_pull_d, next_pymnt_d from new_table""").repartition(1).write\
    .option("header", True)\
    .format("csv")\
    .mode("overwrite")\
    .option("path","/Volumes/datasets/raw/raw_volume/loan_payment_csv").save()

In [0]:
loans_repayment_df = spark.read\
    .format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load("/Volumes/datasets/raw/raw_volume/loan_payment_csv/")


In [0]:
loans_repayment_df.display()


# Loan Defaulter

In [0]:
new_df.printSchema()

In [0]:
spark.sql("""select name_sha2 as member_id, delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,total_rec_late_fee, mths_since_last_delinq, mths_since_last_record from new_table""").display()

In [0]:
spark.sql("""select name_sha2 as member_id, delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,total_rec_late_fee, mths_since_last_delinq, mths_since_last_record from new_table""").repartition(1).write\
    .option("header", True)\
    .format("csv")\
    .mode("overwrite")\
    .option("path","/Volumes/datasets/raw/raw_volume/loan_defaulter_csv").save()

In [0]:
loans_defaulters_df = spark.read\
    .format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load("/Volumes/datasets/raw/raw_volume/loan_defaulter_csv/")

In [0]:
loans_defaulters_df.display()